Este ejercicio utiliza una red neuronal convolucional (CNN) básica para clasificar imágenes RGB. Asegúrate de tener las imágenes organizadas en carpetas correspondientes a sus etiquetas.

In [ ]:
# Importación de librerías
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision import models
import matplotlib.pyplot as plt

In [ ]:
# Definir transformaciones para los datos
transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# Cargar los datos de entrenamiento y validación
train_data = datasets.ImageFolder('path/to/train/data', transform=transform)
val_data = datasets.ImageFolder('path/to/validation/data', transform=transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)

In [ ]:
# Definir el modelo CNN
model = models.resnet18(pretrained=True)  # Usamos ResNet preentrenada
model.fc = nn.Linear(model.fc.in_features, 2)  # Cambiar la capa final para clasificación binaria

In [ ]:
# Definir el optimizador y la función de pérdida
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [ ]:
# Entrenar el modelo
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader)}")

In [ ]:
# Evaluar el modelo
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in val_loader:
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

In [ ]:
print(f'Accuracy on validation set: {100 * correct / total}%')